In [1]:
import requests

url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'
response = requests.get(url)

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("Download complete.")

Download complete.


In [2]:
len(text)

1115394

In [3]:
text[:1000]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [4]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [5]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s ]
decode = lambda l: ''.join([itos[i] for i in l])

print (encode("hii wassup buddy"))
print (decode(encode("hii")))

[46, 47, 47, 1, 61, 39, 57, 57, 59, 54, 1, 40, 59, 42, 42, 63]
hii


In [6]:
import torch
data = torch.tensor(encode(text), dtype = torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [11]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

In [12]:
block_size = 8 # block_size = context length
train_data[:block_size +1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [13]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s ]
decode = lambda l: ''.join([itos[i.item()] for i in l])

In [14]:
y = train_data[1:block_size +1]
x = train_data[:block_size]
for t in range(block_size):
    context = x[:t+1]
    target=y[t]
    print("WORDS: ",decode(context), '->', decode([target]),"\tTOKENS: ", context.tolist(), '->', target.item())
    


WORDS:  F -> i 	TOKENS:  [18] -> 47
WORDS:  Fi -> r 	TOKENS:  [18, 47] -> 56
WORDS:  Fir -> s 	TOKENS:  [18, 47, 56] -> 57
WORDS:  Firs -> t 	TOKENS:  [18, 47, 56, 57] -> 58
WORDS:  First ->   	TOKENS:  [18, 47, 56, 57, 58] -> 1
WORDS:  First  -> C 	TOKENS:  [18, 47, 56, 57, 58, 1] -> 15
WORDS:  First C -> i 	TOKENS:  [18, 47, 56, 57, 58, 1, 15] -> 47
WORDS:  First Ci -> t 	TOKENS:  [18, 47, 56, 57, 58, 1, 15, 47] -> 58


In [15]:
torch.manual_seed(1227)
batch_size = 4;block_size = 8
def get_batch(split):
    data = train_data if split =='train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

xb,yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets: ')
print(yb.shape)
print(yb)

print('--------')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target.item()} ({decode([target])})")


inputs:
torch.Size([4, 8])
tensor([[41, 46,  1, 47, 57,  1, 63, 53],
        [43, 56,  1, 63, 53, 59, 56,  1],
        [63,  1, 46, 47, 57,  1, 42, 43],
        [58,  1, 52, 43, 61, 57,  1, 61]])
targets: 
torch.Size([4, 8])
tensor([[46,  1, 47, 57,  1, 63, 53, 52],
        [56,  1, 63, 53, 59, 56,  1, 40],
        [ 1, 46, 47, 57,  1, 42, 43, 39],
        [ 1, 52, 43, 61, 57,  1, 61, 47]])
--------
when input is [41] the target: 46 (h)
when input is [41, 46] the target: 1 ( )
when input is [41, 46, 1] the target: 47 (i)
when input is [41, 46, 1, 47] the target: 57 (s)
when input is [41, 46, 1, 47, 57] the target: 1 ( )
when input is [41, 46, 1, 47, 57, 1] the target: 63 (y)
when input is [41, 46, 1, 47, 57, 1, 63] the target: 53 (o)
when input is [41, 46, 1, 47, 57, 1, 63, 53] the target: 52 (n)
when input is [43] the target: 56 (r)
when input is [43, 56] the target: 1 ( )
when input is [43, 56, 1] the target: 63 (y)
when input is [43, 56, 1, 63] the target: 53 (o)
when input is [43, 

In [17]:
torch.randint(len(data) - block_size, (batch_size,))


tensor([716643, 778003, 937732, 784120])

In [18]:
print(xb)

tensor([[41, 46,  1, 47, 57,  1, 63, 53],
        [43, 56,  1, 63, 53, 59, 56,  1],
        [63,  1, 46, 47, 57,  1, 42, 43],
        [58,  1, 52, 43, 61, 57,  1, 61]])


In [56]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s ]
decode = lambda l: ''.join([itos[i] for i in l])

print (encode("hii wassup buddy"))
print (decode(encode("hii")))

[46, 47, 47, 1, 61, 39, 57, 57, 59, 54, 1, 40, 59, 42, 42, 63]
hii


In [57]:
import torch;import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size,vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # (B,T,C)
        if targets is None:
            loss=None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets =targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        x = 0
        for _ in range(max_new_tokens):
            x =x+1
            logits, loss = self(idx)
            
            logits = logits[:, -1,:]
            if x==1:
                print(logits.shape)
                print(logits) 
                print(decode(torch.argmax(logits,dim=-1).tolist()))
    
            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(probs,num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1)
        return idx

m = BigramLanguageModel(vocab_size)
out, loss = m(xb,yb)

print(out.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens = 100)[0].tolist()))

torch.Size([32, 65])
tensor(5.0383, grad_fn=<NllLossBackward0>)
torch.Size([1, 65])
tensor([[ 0.1808, -0.0700, -0.3596, -0.9152,  0.6258,  0.0255,  0.9545,  0.0643,
          0.3612,  1.1679, -1.3499, -0.5102,  0.2360, -0.2398, -0.9211,  1.5433,
          1.3488, -0.1396,  0.2858,  0.9651, -2.0371,  0.4931,  1.4870,  0.5910,
          0.1260, -1.5627, -1.1601, -0.3348,  0.4478, -0.8016,  1.5236,  2.5086,
         -0.6631, -0.2513,  1.0101,  0.1215,  0.1584,  1.1340, -1.1539, -0.2984,
         -0.5075, -0.9239,  0.5467, -1.4948, -1.2057,  0.5718, -0.5974, -0.6937,
          1.6455, -0.8030,  1.3514, -0.2759, -1.5108,  2.1048,  2.7630, -1.7465,
          1.4516, -1.5103,  0.8212, -0.2115,  0.7789,  1.5333,  1.6097, -0.4032,
         -0.8345]], grad_fn=<SliceBackward0>)
p

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [58]:
optimizer = torch.optim.AdamW(m.parameters(),lr=1e-3)

In [ ]:
batch_size = 32
epochs = 1000
for steps in range(1000):
    xb,yb = get_batch('train')
    logits, loss = m(xb,yb)
    
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

3.6818594932556152


In [60]:
print(decode(m.generate(idx = torch.zeros((1,1),dtype=torch.long), max_new_tokens = 100)[0].tolist()))

torch.Size([1, 65])
tensor([[ 0.9225, -0.9318, -1.2190, -1.7700, -0.2419, -0.1609,  0.0841, -0.7408,
         -0.5043,  0.3557, -2.2010, -1.3683, -0.6284,  0.3969, -0.4630,  1.3219,
          1.0208, -0.1160,  0.5168,  0.8906, -1.5201,  0.9183,  0.7654,  0.4491,
          0.3367, -1.0883, -0.7855, -0.0162,  0.5197, -0.7268,  0.9879,  1.9390,
          0.0234, -0.3131,  0.4481,  0.6659, -0.7053,  0.8853, -2.0067, -0.4885,
         -0.7453, -1.0473, -0.1490, -1.6790, -1.5129, -0.1367, -0.8163, -0.9988,
          0.7694, -1.2782,  0.5303, -0.6729, -1.5476,  1.2461,  1.8879, -2.1412,
          0.5976, -1.4082,  0.3634, -0.8963, -0.0644,  0.7800,  0.7339, -0.7363,
         -1.6899]], grad_fn=<SliceBackward0>)
S

Wh;;Sq.f ustNzknc;AwgOj$dhPWr,SV?hsusiKpgXXUh;Apmem d?hESXI.i;TrJgkiF-IKbXCAA -botrngFCHAUQkn$

pn$w


In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(epochs), losses)
plt.ylabel("Loss")
plt.xlabel("Epochs")

NameError: name 'plt' is not defined